 # Model Training
 Notebook para entrenar un modelo que va a clasificar las siguientes frutas/verduras/figuras:
 - Platano
 - Jitomate
 - Cubos de colores

 Además de sus diferentes estados de madurez:
 - Inmaduro
 - Medio maduro
 - Maduro
 - Muy maduro

 ## Procesamiento
 El procesamiento se realiza en la nube utilizando los servidores con GPU de Google Colab

Instalamos las librerias

In [1]:
%pip install tensorflow

 Conectamos con Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

 Descomprimimos nuestro dataset en la maquina

In [3]:
!unzip -q "/content/drive/MyDrive/ai/frutasverduras/dataset.zip" -d /content/dataset

 Importamos la paqueteria necesaria

In [4]:
import tensorflow as tf
import os
import matplotlib.pyplot as plt

 Configuramos hiperparámetros
 Las imágenes para MobileNetV2 suelen ser de 224x224 píxeles.

In [5]:
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
DATASET_DIR = '/content/dataset'

 Cargamos el dataset y lo dividimos (90% entrenamiento, 10% validación).

In [6]:
# Datos de entrenamiento
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.1,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Datos de validación (pruebas)
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.1,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Aumentacion de datos
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"), # Volteos aleatorios
    tf.keras.layers.RandomRotation(0.2),                   # Rotación de hasta el 20%
    tf.keras.layers.RandomZoom(0.2),                       # Zoom aleatorio
    tf.keras.layers.RandomTranslation(0.1, 0.1),           # Desplazamientos
    tf.keras.layers.RandomContrast(0.05),                  # Variación de contraste
    tf.keras.layers.RandomBrightness(factor=0.1)           # Variacion de brillo           
])


# Guardamos los nombres de las clases
class_names = train_dataset.class_names
num_classes = len(class_names)
print(f"Clases detectadas ({num_classes}):", class_names)

# Optimizamos la carga de datos en memoria para que la GPU no espere
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)

 Creamos el modelo (Transfer Learning usando MobileNetV2)

In [7]:
# 1. Definimos la capa de preprocesamiento específica para MobileNetV2
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

# 2. Descargamos el modelo base pre-entrenado (sin la capa superior de clasificación)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)

# 3. Congelamos el modelo base para no arruinar los pesos ya aprendidos
base_model.trainable = False

# 4. Construimos nuestro modelo conectando las piezas
inputs = tf.keras.Input(shape=IMG_SIZE + (3,))

x = data_augmentation(inputs) # Aumentamos los datos de entrada
x = preprocess_input(x) # Normaliza los pixeles
x = base_model(x, training=False) # Pasa por MobileNetV2
x = tf.keras.layers.GlobalAveragePooling2D()(x) # Condensa las características
x = tf.keras.layers.Dropout(0.2)(x) # Previene el sobreajuste (overfitting)

# Capa final de predicción (tantas neuronas como clases tengamos)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

 Compilamos y Entrenamos el modelo

In [8]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# Iniciar entrenamiento
EPOCHS = 10 # Las veces que va a recorrer el dataset completo
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS
)

 Evaluamos el modelo

In [ ]:
loss, accuracy = model.evaluate(validation_dataset)
print(f"Precisión del modelo en validación: {accuracy * 100:.2f}%")

Guardamos el modelo en formato keras

In [ ]:
# 'MyDrive' es la carpeta principal de tu Google Drive
model.save('/content/drive/MyDrive/ai/frutasverduras/model.keras')

 Exportamos el modelo a formato TensorFlow Lite (.tflite)

In [ ]:
# Inicializamos el convertidor con nuestro modelo Keras
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Optimizaciones para hacer el modelo más ligero (opcional)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Realizamos la conversión
tflite_model = converter.convert()

# Guardamos el modelo en Google Drive
tflite_save_path = '/content/drive/MyDrive/ai/frutasverduras/model.tflite'
with open(tflite_save_path, 'wb') as f:
    f.write(tflite_model)

print(f"Modelo guardado en: {tflite_save_path}!")

 Guardamos los class_names en un archivo de texto.

In [ ]:
labels_path = '/content/drive/MyDrive/ai/frutasverduras/labels.txt'
with open(labels_path, 'w') as f:
    f.write('\n'.join(class_names))
print(f"Etiquetas guardadas en: {labels_path}")